# WeCo Tutorial: Step-by-Step Well Correlation

This notebook walks through a complete **multi-well stratigraphic correlation** workflow using WeCo.

**Topics covered:**
1. Loading well data
2. Inspecting wells and log curves
3. Configuring correlation parameters
4. Running the correlation engine
5. Interpreting results
6. Data conditioning (Vshale, stacking patterns, electrofacies)
7. Adding constraints (biozones, no-crossing regions)
8. AI-enhanced workflow (quality scoring, uncertainty, anomaly detection)
9. Exporting results (zonation logs, horizon picks)

**Prerequisites:** `pip install weco[ai]` (includes scikit-learn for AI features)

In [ ]:
import os, sys
import numpy as np

# Ensure weco is importable
import weco
from weco.data import WellList, ResFile, ResAndWL
from weco.ext import ProjectExt

print(f"WeCo version: {weco.__version__}")

# Path to bundled example datasets
DATA_DIR = os.path.join(os.path.dirname(weco.__file__), '..', 'data')
DATA_DIR = os.path.abspath(DATA_DIR)
print(f"Data directory: {DATA_DIR}")
print(f"Datasets: {sorted(os.listdir(DATA_DIR))}")

## 1. Loading Well Data

WeCo reads well data from text files containing marker-based well logs.
Each well has:
- **Name**, spatial coordinates (**x, y, z, h**)
- **Continuous data** (e.g., GR, resistivity) — one value per marker
- **Discrete regions** (e.g., facies zones) — intervals with integer IDs

In [ ]:
# Load the simplest dataset: 3 wells, 26 markers, 1 log curve
wells_path = os.path.join(DATA_DIR, 'data_set_1.1', 'wells.txt')
wl = WellList(wells_path)

print(f"Loaded {wl.nbr_wells()} wells from {os.path.basename(wells_path)}")
print()
for w in wl.wells:
    print(f"  {w.name:12s}  markers={w.size:3d}  "
          f"pos=({w.x:.0f}, {w.y:.0f})  "
          f"data={list(w.data.keys())}  "
          f"regions={list(w.region.keys())}")

## 2. Inspecting Well Logs

Let's visualise the log curves to understand what the engine will correlate.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(wl.wells), figsize=(4 * len(wl.wells), 6), sharey=True)
if len(wl.wells) == 1:
    axes = [axes]

colors = ['#2980b9', '#27ae60', '#e74c3c', '#f39c12', '#8e44ad']

for i, (well, ax) in enumerate(zip(wl.wells, axes)):
    color = colors[i % len(colors)]
    ax.set_title(well.name, fontweight='bold', color=color)
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)
    
    depth = list(range(well.size))  # marker index as depth
    
    for dname in sorted(well.data.keys()):
        if dname.upper() in ('DEPTH', 'MD', 'TVD'):
            continue
        vals = list(well.data[dname])
        ax.plot(vals[:well.size], depth[:well.size], color=color, label=dname)
    
    if i == 0:
        ax.set_ylabel('Marker Index')
    ax.legend(fontsize=8)

fig.suptitle('Well Log Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Basic Correlation

The simplest correlation uses only the **variance cost** — it tries to match
markers with similar log values across wells.

Key parameters:
- `var-data` — which log curve to correlate on
- `max-cor` — search width (quality vs speed tradeoff)
- `out-nbr-cor` — how many alternative solutions to keep

In [ ]:
import time

# Create a project and set minimal options
proj = ProjectExt()
proj.set_option_ext('cost-function', 'composite')
proj.set_option_ext('var-data', 'VarData1')   # use the first log
proj.set_option_ext('var-weight', 1.0)
proj.set_option_ext('max-cor', 50)             # search width
proj.set_option_ext('out-nbr-cor', 5)          # keep top-5 solutions

# Run the correlation
t0 = time.perf_counter()
proj.run(wells_path)
elapsed = (time.perf_counter() - t0) * 1000

# Get results
res = proj.get_res_file()
rw = ResAndWL(res, wl)

print(f"Correlation completed in {elapsed:.1f} ms")
print(f"Number of alternative correlations: {res.get_nbr_results()}")
print()
for i in range(min(5, res.get_nbr_results())):
    path = res.get_result_full_path(i)
    print(f"  #{i}: cost = {res.get_result_cost(i):.6f}, nodes = {len(path)}")

## 4. Visualising Correlation Results

Let's plot the best correlation — horizontal lines connect matched markers across wells.

In [ ]:
def plot_correlation(well_list, res_file, cor_num=0, title=''):
    """Plot a correlation result with tie lines between wells."""
    path = res_file.get_result_full_path(cor_num)
    cost = res_file.get_result_cost(cor_num)
    wells = well_list.wells
    n_wells = len(wells)
    
    fig, axes = plt.subplots(1, n_wells, figsize=(4 * n_wells, 8), sharey=True)
    if n_wells == 1:
        axes = [axes]
    
    # Positions for wells (evenly spaced)
    x_pos = list(range(n_wells))
    
    for i, (well, ax) in enumerate(zip(wells, axes)):
        color = colors[i % len(colors)]
        ax.set_title(well.name, fontweight='bold', color=color)
        ax.invert_yaxis()
        ax.grid(True, alpha=0.2)
        
        # Plot first data curve
        for dname in sorted(well.data.keys()):
            if dname.upper() in ('DEPTH', 'MD', 'TVD'):
                continue
            vals = list(well.data[dname])[:well.size]
            ax.plot(vals, range(len(vals)), color=color, linewidth=1)
            break
        if i == 0:
            ax.set_ylabel('Marker Index')
    
    # Draw correlation lines on a shared figure
    fig_trans = fig.transFigure.inverted()
    step = max(1, len(path) // 30)  # subsample for readability
    for k in range(0, len(path), step):
        node = path[k]
        for i in range(n_wells - 1):
            # Get display coordinates for marker positions
            p1 = axes[i].transData.transform((0, node[i]))
            p2 = axes[i+1].transData.transform((0, node[i+1]))
            p1_fig = fig.transFigure.inverted().transform(p1)
            p2_fig = fig.transFigure.inverted().transform(p2)
            line = plt.Line2D(
                [p1_fig[0], p2_fig[0]], [p1_fig[1], p2_fig[1]],
                transform=fig.transFigure, color='gray',
                alpha=0.4, linewidth=0.5)
            fig.lines.append(line)
    
    fig.suptitle(f'{title}Correlation #{cor_num}  (cost={cost:.4f})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

plot_correlation(wl, res, cor_num=0, title='Basic Variance — ')

## 5. Comparing Alternative Correlations

WeCo returns the n-best solutions. Comparing them reveals where the correlation is robust
(alternatives agree) vs uncertain (alternatives differ).

In [ ]:
# Compare the best and 2nd-best by looking at where they differ
if res.get_nbr_results() >= 2:
    path0 = res.get_result_full_path(0)
    path1 = res.get_result_full_path(1)
    
    n_diff = sum(1 for a, b in zip(path0, path1) if a != b)
    print(f"Best vs 2nd-best: {n_diff}/{len(path0)} nodes differ")
    print(f"Cost difference: {res.get_result_cost(1) - res.get_result_cost(0):.6f}")
    
    if n_diff > 0:
        print("\nDiffering nodes (node_idx: best → 2nd best):")
        shown = 0
        for k, (a, b) in enumerate(zip(path0, path1)):
            if a != b:
                print(f"  Node {k}: {a} → {b}")
                shown += 1
                if shown >= 10:
                    print(f"  ... and {n_diff - shown} more")
                    break
else:
    print("Only one solution found — try increasing max-cor for more alternatives.")

## 6. Data Conditioning

Before correlating, we can enrich the well data with derived curves:
- **Vshale** — clay volume from GR (normalised)
- **Stacking pattern** — GR derivative (fining-up vs coarsening-up)
- **Electrofacies** — cluster-based facies from multiple logs

In [ ]:
from weco.preprocessing import (
    compute_vshale, compute_stacking_pattern, normalise_log,
    compute_electrofacies
)

# Reload wells for a fresh start
wl2 = WellList(wells_path)

# Find the primary log name
log_name = sorted(wl2.wells[0].data.keys())[0]
print(f"Primary log: {log_name}")

# Apply transforms
normalise_log(wl2, log_name, output_name=f"{log_name}_norm")
for w in wl2.wells:
    compute_vshale(w, gr_name=log_name)
    compute_stacking_pattern(w, gr_name=log_name)

# Show new channels
print(f"\nData channels after conditioning:")
for w in wl2.wells:
    print(f"  {w.name}: {sorted(w.data.keys())}")

## 7. Multi-Log Correlation

Using multiple log curves improves correlation quality — the engine matches
the *combined signature* rather than just one curve.

In [ ]:
# Use a dataset with multiple log curves
ds12_path = os.path.join(DATA_DIR, 'data_set_1.2', 'wells.txt')
if os.path.exists(ds12_path):
    wl_multi = WellList(ds12_path)
    print(f"Dataset 1.2: {wl_multi.nbr_wells()} wells")
    print(f"Data channels: {sorted(wl_multi.wells[0].data.keys())}")
    print(f"Regions: {sorted(wl_multi.wells[0].region.keys())}")
    
    # Load options from the dataset
    opts_path = os.path.join(DATA_DIR, 'data_set_1.2', 'option.txt')
    proj2 = ProjectExt()
    proj2.option_load(os.path.abspath(opts_path))
    
    t0 = time.perf_counter()
    proj2.run(os.path.abspath(ds12_path))
    elapsed = (time.perf_counter() - t0) * 1000
    
    res2 = proj2.get_res_file()
    print(f"\nCorrelation: {elapsed:.1f} ms, {res2.get_nbr_results()} solutions")
    for i in range(min(3, res2.get_nbr_results())):
        print(f"  #{i}: cost = {res2.get_result_cost(i):.6f}")
else:
    print("Dataset 1.2 not found — skipping multi-log example.")

## 8. Adding Constraints

Constraints enforce geological rules:
- **no_crossing** — correlation lines cannot cross region boundaries (e.g., biozones)
- **same_region** — prefer correlating within the same region (soft constraint)
- **gap_cost** — control how expensive it is to skip markers (hiatuses)

In [ ]:
# Run with constraints vs without, compare cost
proj_no_constr = ProjectExt()
proj_no_constr.set_option_ext('cost-function', 'composite')
proj_no_constr.set_option_ext('var-data', 'VarData1')
proj_no_constr.set_option_ext('var-weight', 1.0)
proj_no_constr.set_option_ext('max-cor', 50)
proj_no_constr.set_option_ext('out-nbr-cor', 3)
proj_no_constr.run(wells_path)
res_nc = proj_no_constr.get_res_file()

proj_constr = ProjectExt()
proj_constr.set_option_ext('cost-function', 'composite')
proj_constr.set_option_ext('var-data', 'VarData1')
proj_constr.set_option_ext('var-weight', 1.0)
proj_constr.set_option_ext('max-cor', 50)
proj_constr.set_option_ext('out-nbr-cor', 3)
proj_constr.set_option_ext('const-gap-cost', 0.3)  # penalise gaps
proj_constr.run(wells_path)
res_c = proj_constr.get_res_file()

print("Without gap cost:")
p0 = res_nc.get_result_full_path(0)
print(f"  Best cost: {res_nc.get_result_cost(0):.6f}, nodes: {len(p0)}")

print("With gap cost = 0.3:")
p1 = res_c.get_result_full_path(0)
print(f"  Best cost: {res_c.get_result_cost(0):.6f}, nodes: {len(p1)}")

## 9. Saving and Loading Results

Results can be saved as native WeCo format and reloaded later.

In [ ]:
import tempfile

# Save the result file
with tempfile.NamedTemporaryFile(suffix='.txt', delete=False, mode='w') as f:
    tmp_path = f.name

res.write(tmp_path)
print(f"Saved result to: {tmp_path}")

# Show first 15 lines of the file
with open(tmp_path) as f:
    for i, line in enumerate(f):
        if i >= 15:
            print('...')
            break
        print(line, end='')

# Reload and verify
res_reloaded = ResFile(tmp_path)
print(f"\nReloaded: {res_reloaded.get_nbr_results()} correlations")
print(f"Original best cost:  {res.get_result_cost(0):.10f}")
print(f"Reloaded best cost:  {res_reloaded.get_result_cost(0):.10f}")

os.unlink(tmp_path)

## 10. AI-Enhanced Workflow

WeCo includes AI modules for quality scoring, uncertainty quantification,
and anomaly detection.

In [ ]:
try:
    from weco.ai.quality import CorrelationQuality
    from weco.ai.uncertainty import CorrelationUncertainty
    from weco.ai.anomaly import CorrelationAnomalyDetector, StatisticalAnomalyDetector
    HAS_AI = True
    print("AI modules loaded successfully")
except ImportError as e:
    HAS_AI = False
    print(f"AI modules not available: {e}")
    print("Install with: pip install weco[ai]")

In [ ]:
if HAS_AI:
    # Quality scoring
    scorer = CorrelationQuality()
    scores = scorer.score(res, wl)
    print("Quality scores per correlation:")
    for i, s in enumerate(scores[:5]):
        print(f"  #{i}: {s:.4f}")
    
    print()
    
    # Uncertainty from n-best ensemble
    uc = CorrelationUncertainty()
    uncertainty = uc.compute(res, wl)
    print(f"Uncertainty analysis (n-best={min(5, res.get_nbr_results())} paths):")
    for key, val in uncertainty.items():
        if isinstance(val, (int, float)):
            print(f"  {key}: {val:.4f}")
else:
    print("Skipping AI features (scikit-learn not installed)")

## 11. Exporting for Geomodelling

WeCo can export results as:
- **Zonation logs** — per-well zone assignments for facies modelling
- **Horizon picks** — depth of each horizon in each well for structural modelling
- **CSV/JSON** — for any downstream application

In [ ]:
from weco.export import (
    res_to_zonation_log,
    res_to_horizon_picks,
    correlation_summary,
)

# Zonation log
zonation = res_to_zonation_log(res, wl, cor_num=0)
print("Zonation logs:")
for well_name, info in zonation.items():
    print(f"  {well_name}: {info['n_zones']} zones, "
          f"{len(info['zone'])} markers")

# Horizon picks
picks = res_to_horizon_picks(res, wl, cor_num=0)
print(f"\nHorizon picks: {len(picks)} horizons")
for i, pick in enumerate(picks[:5]):
    print(f"  Horizon {i}: {pick}")

# Summary
summary = correlation_summary(res, wl, n_best=3)
print(f"\nSummary keys: {list(summary.keys())}")

## 12. Transport Direction & Distality

For basins with a dominant sediment transport direction, WeCo can compute
per-well distality (proximal → distal gradient) and use it as an additional
cost function.

In [ ]:
from weco.distality import compute_distality, best_transport_direction

# Find the best transport direction for our well set
best_az, best_range = best_transport_direction(wl)
print(f"Best transport azimuth: {best_az:.1f} degrees (N)")
print(f"Distality range at best azimuth: {best_range:.4f}")

# Compute distality for each well
wl_dist = WellList(wells_path)  # fresh copy
compute_distality(wl_dist, best_az)

print("\nPer-well distality:")
for w in wl_dist.wells:
    if 'distality' in w.data:
        d = w.data['distality'][0]
        print(f"  {w.name}: distality = {d:.4f}")
    elif 'distality' in w.region:
        d = w.region['distality'][0] if w.region['distality'] else '?'
        print(f"  {w.name}: distality region = {d}")

## 13. Custom Cost Functions

WeCo's plugin architecture allows adding custom cost functions in Python.
Here's how to use the built-in `BiozonAgeCost`:

In [ ]:
from weco.cost_functions import BiozonAgeCost, FaciesGroupCost, TransportDirectionCost

# These are CCFPartExt subclasses that can be added to a project:
print("Available cost function plugins:")
print(f"  BiozonAgeCost      — penalises cross-biozone correlations")
print(f"  FaciesGroupCost    — lateral equivalence groups")
print(f"  TransportDirectionCost — distality gradient consistency")
print()
print("Usage:")
print("  proj = ProjectExt()")
print("  BiozonAgeCost.ZONE_AGES = {1: 54.0, 2: 51.5, 3: 48.0}")
print("  proj.add_ccf_part(BiozonAgeCost)")
print("  proj.run('wells.txt')")

## 14. Hierarchical (Multi-Scale) Correlation

WeCo supports **hierarchical correlation** (§8) where large-scale
stratigraphic boundaries are correlated first, then fine-scale markers
are correlated within each coarse interval.

This mimics geological practice: correlate sequence boundaries first,
then correlate parasequences within each systems tract.

### Theory

1. **Coarse pass** — correlate only *key surfaces* (e.g. flooding surfaces)
   using `multiscale_level=1`.
2. **Fine pass** — fix the coarse tie-points and correlate detailed markers
   within each interval using `multiscale_level=2`.

The coarse pass provides structural constraints that prevent impossible
cross-overs at the fine scale.

In [ ]:
# §14.1 Coarse-scale correlation (key boundaries only)
from weco.ext import ProjectExt

proj_coarse = ProjectExt()
proj_coarse.set_options_ext(
    var_data=os.path.join(DATA_DIR, 'data_set_2', 'data'),
    cost_function='composite',
    max_cor=1,
    multiscale_level=1,  # coarse pass
)
proj_coarse.run(os.path.join(DATA_DIR, 'data_set_2', 'wells.txt'))
res_coarse = proj_coarse.get_res_file()
print(f'Coarse correlation: {res_coarse.get_nbr_result()} results, '
      f'cost = {res_coarse.get_cost(0):.4f}')

In [ ]:
# §14.2 Fine-scale correlation (within coarse intervals)
proj_fine = ProjectExt()
proj_fine.set_options_ext(
    var_data=os.path.join(DATA_DIR, 'data_set_2', 'data'),
    cost_function='composite',
    max_cor=3,
    multiscale_level=2,  # fine pass
    band_width=20,       # Sakoe-Chiba band for efficiency
)
proj_fine.run(os.path.join(DATA_DIR, 'data_set_2', 'wells.txt'))
res_fine = proj_fine.get_res_file()
print(f'Fine correlation: {res_fine.get_nbr_result()} results, '
      f'cost = {res_fine.get_cost(0):.4f}')

In [ ]:
# §14.3 Compare coarse vs fine results
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].set_title('Coarse-scale correlation')
plot_correlation(proj_coarse.get_well_list(), res_coarse, cor_num=0,
                 title='Coarse (key surfaces)')
axes[1].set_title('Fine-scale correlation')
plot_correlation(proj_fine.get_well_list(), res_fine, cor_num=0,
                 title='Fine (all markers)')
plt.tight_layout()
plt.show()
print('\nHierarchical approach ensures fine-scale ties respect '
      'sequence boundaries.')

## Summary

This tutorial covered the core WeCo workflow:

| Step | Tool | Purpose |
|------|------|---------|
| Load data | `WellList(path)` | Read wells with logs and regions |
| Configure | `ProjectExt.set_option_ext()` | Set cost weights, constraints |
| Condition | `preprocessing.*` | Derive Vshale, stacking, electrofacies |
| Run | `ProjectExt.run()` | Execute graph-DTW correlation |
| Inspect | `ResFile.get_result_full_path()` | View correlation paths |
| Score | `ai.quality.CorrelationQuality` | Quality and uncertainty |
| Export | `export.res_to_zonation_log()` | Zonation logs, horizon picks |
| Save | `ResFile.write()` | Persist results to disk |

For an interactive GUI experience, launch **WeCo Studio** with:
```bash
WeCoStudio
```

For headless/API usage:
```bash
uvicorn weco.api:app --host 0.0.0.0 --port 8000
```